In [ ]:
"""
Name: Module 2 Chicago Taxi September Trip.py
Course: Data Preparation and Analysis
Created Date: November 1, 2023
Author: Ming-Long Lam, Ph.D.
Organization: Illinois Institute of Technology
"""

In [ ]:
import matplotlib.pyplot as plt
import numpy
import pandas
import seaborn
import sys

In [ ]:
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter)
from scipy.stats import chi2, t
from scipy.stats import pearsonr

In [ ]:
# Set some options for printing all the columns
numpy.set_printoptions(precision = 10, threshold = sys.maxsize)
numpy.set_printoptions(linewidth = numpy.inf)

In [ ]:
pandas.set_option('display.max_columns', None)
pandas.set_option('display.expand_frame_repr', False)
pandas.set_option('max_colwidth', None)

In [ ]:
pandas.options.display.float_format = '{:,.10f}'.format

In [ ]:
def PearsonChiSquareTest (xCat, yCat):
   '''Perform the Pearson test of independence and compute the Cramer V

   Argument:
   ---------
   xCat : a Pandas Series of categorical observations
   yCat : another Pandas Series of categorical observations

   Output:
   -------
   xNCat     : Number of X categories
   yNCat     : Number of Y categories
   chiSqStat : Test statistic
   chiSqDf   : Test degrees of freedom
   chiSqSig  : Test significance
   cramerv   : Cramer V
   '''

   # Generate the crosstabulation
   obsCount = pandas.crosstab(index = xCat, columns = yCat, margins = False, dropna = True)
   xNCat = obsCount.shape[0]
   yNCat = obsCount.shape[1]
   cTotal = obsCount.sum(axis = 1)
   rTotal = obsCount.sum(axis = 0)
   nTotal = numpy.sum(rTotal)
   expCount = numpy.outer(cTotal, (rTotal / nTotal))

   # Calculate the Chi-Square statistics
   chiSqStat = ((obsCount - expCount)**2 / expCount).to_numpy().sum()
   chiSqDf = (xNCat - 1) * (yNCat - 1)
   if (chiSqDf > 0):
      chiSqSig = chi2.sf(chiSqStat, chiSqDf)
      cramerv = chiSqStat / nTotal / (min(xNCat, yNCat) - 1.0)
      cramerv = numpy.sqrt(cramerv)
   else:
      chiSqSig = numpy.NaN
      cramerv = numpy.NaN

   outlist = [xNCat, yNCat, chiSqStat, chiSqDf, chiSqSig, cramerv]
   return (outlist)

In [ ]:
def AnalysisOfVarianceTest (xCat, yCont):
   '''Perform the Pearson test of independence and compute the Cramer V

   Argument:
   ---------
   xCat  : a Pandas Series of categorical observations
   yCont : a Pandas Series of continuous observations

   Output:
   -------
   nGroup : Number of X categories
   etasq  : Eta-Squared value
   '''

   df = pandas.DataFrame(columns = ['x', 'y'])
   df['x'] = xCat
   df['y'] = yCont

   # Total Count and Sum of Squares
   totalCount = df['y'].count()
   totalSSQ = df['y'].var(ddof = 0) * totalCount

   # Within Group Count and Sums of Squares
   groupCount = df.groupby('x').count()
   groupSSQ = df.groupby('x').var(ddof = 0) * groupCount
   nGroup = groupCount.shape[0]

   withinSSQ = numpy.sum(groupSSQ.values)

   if (totalSSQ > 0.0):
      etasq = 1.0 - withinSSQ / totalSSQ
   else:
      etasq = numpy.NaN
       
   outlist = [nGroup, etasq]
   return (outlist)

In [ ]:
taxi_m9 = pandas.read_csv('C:\\IIT\\Data Preparation and Analysis\\Data\\Chicago_September_taxi.csv')

In [ ]:
# Map the community area number to its name
def major_community (area):
    if (area == 6):
        name = 'Lake View'
    elif (area == 7):
        name = 'Lincoln Park'
    elif (area == 8):
        name = 'Near North Side'
    elif (area == 28):
        name = 'Near West Side'
    elif (area == 32):
        name = 'Loop'
    elif (area == 33):
        name = 'Near South Side'
    elif (area == 56 or area == 64):
        name = 'Midway Airport'
    elif (area == 76):
        name = "O'Hare Airport"
    else:
        name = 'Other Community'

    return (name)

In [ ]:
# Create the pickup community names and the dropoff community names
pickup_area = taxi_m9['Pickup Community Area'].apply(major_community)
dropoff_area = taxi_m9['Dropoff Community Area'].apply(major_community)

In [ ]:
# Frequency of pickup and dropoff community names
pickup_count = pickup_area.value_counts()
dropoff_count = dropoff_area.value_counts()

In [ ]:
# Crosstabulate pickup area versus dropoff communities
count_table = pandas.crosstab(pickup_area, dropoff_area)
fig, ax = plt.subplots(figsize = (10,8), dpi = 200)
seaborn.heatmap(count_table, annot = False,  linewidths = 0.1, ax = ax, cmap = 'coolwarm',
                cbar_kws = {'label': 'Count'})
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Calculate dropoff community percents PER pickup community
percent_table = 100.0 * count_table.div(count_table.sum(axis = 'columns'), axis = 'index')
fig, ax = plt.subplots(figsize = (10,8), dpi = 200)
seaborn.heatmap(percent_table, annot = False,  linewidths = 0.1, ax = ax, cmap = 'Spectral',
                cbar_kws = {'label': 'Row Percent'})
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Generate a horizontal stacked percentage bar chart
cmap = ['indianred','sandybrown','royalblue', 'olivedrab','purple', 'bisque', 'green', 'orange', 'gray']
barThick = 0.8
yCat = percent_table.columns
accPct = numpy.zeros(percent_table.shape[0])
fig, ax = plt.subplots(figsize = (12,6), dpi = 200)
for j in range(len(yCat)):
   catLabel = yCat[j]
   plt.barh(percent_table.index, percent_table[catLabel], color = cmap[j], left = accPct, label = catLabel, height = barThick)
   accPct = accPct + percent_table[catLabel]
ax.xaxis.set_major_locator(MultipleLocator(base = 10))
ax.xaxis.set_major_formatter(FormatStrFormatter('%.0f%%'))
ax.xaxis.set_minor_locator(MultipleLocator(base = 5))
ax.set_xlabel('Percent of Dropoff Community')
ax.set_ylabel('Pickup Community')
plt.grid(axis = 'x')
plt.legend(title = 'Dropoff Community', loc = 'lower center', bbox_to_anchor = (1.1, 0.4), ncol = 1)
plt.show()

In [ ]:
# Generate a scatterplot of Trip Miles versus Trip Time
trip_minute = taxi_m9['Trip Seconds'].apply(lambda x: x/60)
trip_minute.name = 'Trip Minute'
trip_speed = taxi_m9['Trip Miles'] / (trip_minute / 60)
trip_speed.name = 'Trip Speed'

In [ ]:
# Histogram of Trip Miles
plt.figure(figsize = (8,6), dpi = 200)
h_mile = plt.hist(taxi_m9['Trip Miles'], bins = range(0,100,10), color = 'seagreen')
plt.xlabel('Trip Miles')
plt.ylabel('Number of Trips')
plt.xticks(range(0,100,10))
plt.yticks(range(0,160000,20000))
plt.grid(axis = 'y', linestyle = '--')
plt.show()

In [ ]:
# Histogram of Trip Minutes
plt.figure(figsize = (8,6), dpi = 200)
h_minute = plt.hist(trip_minute, bins = range(0,195,15), color = 'royalblue')
plt.xlabel('Trip Minutes')
plt.ylabel('Number of Trips')
plt.xticks(range(0,195,15))
plt.yticks(range(0,110000,10000))
plt.grid(axis = 'y', linestyle = '--')
plt.show()

In [ ]:
# Histogram of Trip Speed
plt.figure(figsize = (8,6), dpi = 200)
h_speed = plt.hist(trip_speed, bins = range(0,110,10), color = 'indianred')
plt.xlabel('Trip Speed (Miles Per Hour)')
plt.ylabel('Number of Trips')
plt.xticks(range(0,110,10))
plt.yticks(range(0,130000,10000))
plt.grid(axis = 'y', linestyle = '--')
plt.show()

In [ ]:
# Trip Miles versus Trip Minutes, colored by Trip Speed
fig, ax = plt.subplots(figsize = (8,6), dpi = 200)
im = ax.scatter(trip_minute, taxi_m9['Trip Miles'], c = trip_speed, s = 10)
ax.xaxis.set_major_locator(MultipleLocator(base = 30))
ax.xaxis.set_minor_locator(MultipleLocator(base = 15))
ax.yaxis.set_major_locator(MultipleLocator(base = 10))
ax.yaxis.set_minor_locator(MultipleLocator(base = 5))
ax.axline((60,30), slope = 0.5, linestyle = '--', color = 'red')
ax.set_xlabel('Trip Minutes')
ax.set_ylabel('Trip Miles')
plt.colorbar(im, ax = ax, label = 'Trip Speed (Miles Per Hour)', orientation = 'horizontal')
plt.grid(axis = 'both')
plt.show()

In [ ]:
plot_data = taxi_m9['Payment Type'].value_counts()
payment_type = list(plot_data.index)

In [ ]:
plt.figure(figsize = (8,6), dpi = 200)
plt.bar(payment_type, plot_data, color = 'limegreen')
plt.xlabel('Payment Method')
plt.ylabel('Number of Trips')
plt.yticks(range(0,100000,10000))
plt.grid(axis = 'y', linestyle = '--')
plt.show()

In [ ]:
# Sort the payment type in ascending order of median fare
u = taxi_m9['Payment Type'].astype('category').copy()
group_median = taxi_m9.groupby('Payment Type')['Fare'].median().sort_values(ascending = True)
taxi_m9['Payment Type'] = u.cat.reorder_categories(group_median.index)

In [ ]:
fig, ax = plt.subplots(figsize = (8,6), dpi = 200)
taxi_m9.boxplot(column = 'Fare', by = 'Payment Type', vert = False, ax = ax)
ax.set_xlabel('Fare')
ax.set_ylabel('Payment Type')
ax.xaxis.set_major_locator(MultipleLocator(base = 20))
ax.xaxis.set_minor_locator(MultipleLocator(base = 10))
plt.title('')
plt.suptitle('')
plt.grid(axis = 'y')
plt.show()

In [ ]:
# Summary of tips by payment type
summary_tips = taxi_m9.groupby('Payment Type')['Tips'].describe()
print('=== Summary of Tips by Payment Type ===')
print(summary_tips)

In [ ]:
# Percentage of Trip Tip based on Trip Fare
tip_percent = numpy.where(taxi_m9['Fare'] > 0.0, 100 * (taxi_m9['Tips'] / taxi_m9['Fare']), numpy.nan)
taxi_m9['Percent Tips'] = numpy.copy(tip_percent)

In [ ]:
# Summary of tip percentage by payment type
summary_percent_tips = taxi_m9.groupby('Payment Type')['Percent Tips'].describe()
print('=== Summary of Tip Percentage by Payment Type ===')
print(summary_percent_tips)

In [ ]:
fig, ax = plt.subplots(figsize = (8,6), dpi = 200)
taxi_m9.boxplot(column = 'Percent Tips', by = 'Payment Type', vert = False, ax = ax)
ax.set_xlabel('Tips as Percentage of Fare')
ax.set_ylabel('Payment Type')
ax.xaxis.set_major_locator(MultipleLocator(base = 100))
ax.xaxis.set_minor_locator(MultipleLocator(base = 50))
plt.title('')
plt.suptitle('')
plt.grid(axis = 'y')
plt.show()

In [ ]:
# Detect and measure association between Pickup Community and Dropoff Community
outlist = PearsonChiSquareTest (pickup_area, dropoff_area)
xNCat = outlist[0]
yNCat = outlist[1]
chiSqStat = outlist[2]
chiSqDf = outlist[3]
chiSqSig = outlist[4]
cramerv = outlist[5]

In [ ]:
# Detect and measure association between Trip Minutes and Trip Miles
pearson_r = pearsonr (trip_minute, taxi_m9['Trip Miles'])
print('Pearson Correlation = {:,.8f}'.format(pearson_r[0]))
print('Test Significance = {:,.8e}'.format(pearson_r[1]))

In [ ]:
# Alternatively verfiy the test significance value
tDF = taxi_m9.shape[0] - 2
se_pcorr = (1 - pearson_r[0]) / tDF
se_pcorr = numpy.sqrt(se_pcorr)
t_stat = pearson_r[0] / se_pcorr
t_pvalue = 2.0 * t.sf(abs(t_stat), tDF)

In [ ]:
# Scatterplot of trip miles by trip minutes again
fig, ax = plt.subplots(figsize = (8,6), dpi = 200)
im = ax.scatter(trip_minute, taxi_m9['Trip Miles'], s = 10, color = 'royalblue')
ax.xaxis.set_major_locator(MultipleLocator(base = 30))
ax.xaxis.set_minor_locator(MultipleLocator(base = 15))
ax.yaxis.set_major_locator(MultipleLocator(base = 10))
ax.yaxis.set_minor_locator(MultipleLocator(base = 5))
ax.axline((60,30), slope = 0.5, linestyle = '--', color = 'red')
ax.set_xlabel('Trip Minutes')
ax.set_ylabel('Trip Miles')
plt.title('Original ' + str(taxi_m9.shape[0]) + ' Observztions')
plt.grid(axis = 'both')
plt.show()

In [ ]:
# Randomly select 5% of observations
from sklearn.model_selection import train_test_split
train_df, test_df =  train_test_split(taxi_m9[['Trip Miles']].join(trip_minute), train_size = 0.05,
                                      random_state = 2023571) 

In [ ]:
# Sanity check the scatterplot
fig, ax = plt.subplots(figsize = (8,6), dpi = 200)
im = ax.scatter(train_df['Trip Minute'], train_df['Trip Miles'], s = 10, color = 'green')
ax.xaxis.set_major_locator(MultipleLocator(base = 30))
ax.xaxis.set_minor_locator(MultipleLocator(base = 15))
ax.yaxis.set_major_locator(MultipleLocator(base = 10))
ax.yaxis.set_minor_locator(MultipleLocator(base = 5))
ax.axline((60,30), slope = 0.5, linestyle = '--', color = 'red')
ax.set_xlabel('Trip Minutes')
ax.set_ylabel('Trip Miles')
plt.title('A 5% Random Sample')
plt.grid(axis = 'both')
plt.show()

In [ ]:
print('=== A 5% Random Sample ===')

In [ ]:
pearson_r5 = pearsonr (train_df['Trip Minute'], train_df['Trip Miles'])
print('Pearson Correlation = {:,.8f}'.format(pearson_r5[0]))
print('Test Significance = {:,.8f}'.format(pearson_r5[1]))

In [ ]:
# Calculate distance correlation by calling the dcor() function in the R package energy
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

In [ ]:
energy = importr("energy")
with (robjects.default_converter + pandas2ri.converter).context():
  r_df = robjects.conversion.get_conversion().py2rpy(train_df)

In [ ]:
distance_corr = energy.dcor(r_df[0], r_df[1])
print('Distance Correlation = {:,.8f}'.format(distance_corr[0]))

In [ ]:
# Detect and measure association between Payment Type and Tips
group_stat = taxi_m9[['Payment Type', 'Percent Tips']].groupby('Payment Type').describe()
group_stat = group_stat['Percent Tips'].sort_values(by = ['mean'], ascending = False)

In [ ]:
outlist = AnalysisOfVarianceTest (taxi_m9['Payment Type'], taxi_m9['Percent Tips'])
nGroup = outlist[0]
etasq = outlist[1]